# RSNA Knee Abnormality Detection — Step 1: Explore

Goal of this notebook: **find out what the data actually looks like** before writing any model code.

Run cells top to bottom. Runtime > Change runtime type > CPU is fine here (no GPU needed for exploration).

## 1. Get the code

In [ ]:
# If you've pushed this repo to GitHub, clone it. Otherwise skip and upload src/ manually.
REPO_URL = ""  # e.g. "https://github.com/yourname/rsna-knee.git"

import os
if REPO_URL:
    !git clone -q $REPO_URL rsna-knee
    os.chdir("rsna-knee")
!ls

## 2. Install deps

In [ ]:
!pip install -q pydicom kaggle
import pydicom, pandas as pd
print("pydicom", pydicom.__version__, "| pandas", pd.__version__)

## 3. Kaggle credentials

Get `kaggle.json` from https://www.kaggle.com/settings > API > **Create New Token**, then upload it below.
Don't paste the token contents anywhere else — the upload keeps it inside your own Colab session.

In [ ]:
from google.colab import files
import os, json

if not os.path.exists("/root/.kaggle/kaggle.json"):
    files.upload()  # select kaggle.json
    os.makedirs("/root/.kaggle", exist_ok=True)
    os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("credentials in place")

## 4. How big is this thing?

**Check before downloading.** Colab gives you ~80–110 GB of disk. If the dataset is larger than that,
downloading it here is a dead end and you should work in a Kaggle notebook instead (data is pre-mounted
there, no download, and the final submission has to run as a Kaggle notebook anyway).

In [ ]:
COMP = "rsna-knee-abnormality-detection"
!kaggle competitions files -c $COMP
!df -h /content | tail -1

## 5. Download

Start with the small metadata/CSV files only — those alone answer most of the important questions
(label prevalence, how reports are stored, train/test split size) and take seconds instead of hours.

In [ ]:
import os
os.makedirs(f"/content/{COMP}", exist_ok=True)

# Pull individual small files first. Adjust names to match the output of step 4.
for fname in ["train.csv", "sample_submission.csv"]:
    !kaggle competitions download -c $COMP -f $fname -p /content/$COMP

!cd /content/$COMP && (unzip -o -q '*.zip' 2>/dev/null; ls -la)

In [ ]:
# Full download — only run this once you know it fits (step 4).
# !kaggle competitions download -c $COMP -p /content
# !unzip -q /content/$COMP.zip -d /content/$COMP

## 6. Explore

In [ ]:
os.environ["RSNA_KNEE_DATA"] = f"/content/{COMP}"
!python src/explore.py

In [ ]:
# Once image files are downloaded, add these:
# !python src/explore.py --dicom --series

## 7. Sanity-check the submission format

Produces a valid all-0.5 submission. Scores ~0.5 AUC, but proves the file format is right
before you spend nine hours of runtime on a real model.

In [ ]:
!python src/submission.py --constant 0.5 -o submission.csv
import pandas as pd; pd.read_csv("submission.csv").head()

---
## Next

Paste the output of step 6 back into Claude Code. The dataset layout it reveals — how series are nested,
how many slices per exam, whether reports ship as a CSV column or separate files — determines how
`dataset.py`, `model.py`, and `train.py` get written. Those are deliberately not written yet, because
guessing the layout would mean rewriting them anyway.